# RQ4: Limitations

**Research Question**: What are the causes of unsuccessful generalization attempts?

This notebook analyzes the limitations of test generalization by examining filtering causes and processing failures:
- **Exclusion Analysis**: Overall filtering of tests, assertions, and generalizations by variant
- **Filter Effectiveness**: Detailed breakdown of which filters cause exclusions
- **SPF Failures**: Symbolic PathFinder execution error categorization
- **Test Failures**: Runtime test execution failures by variant
- **Pipeline Analysis**: Processing stage failures in extended dataset evaluation

In [ ]:
from teralizer.config import db_config
from teralizer.rq4_limitations import (
    get_exclusions_summary_data,
    get_filtering_exclusions_data,
    get_spf_failures_data,
    get_test_executions_by_variant_data,
    get_test_failures_by_projects_data,
    get_processing_failures_by_cause_data,
    compute_exclusion_percentages,
    compute_filtering_exclusions_summary,
    compute_spf_error_categorization,
    compute_test_failures_by_variant_summary,
    compute_test_failures_by_projects_summary,
    compute_processing_failures_by_stage_and_cause,
    generate_exclusions_summary_table,
    generate_filtering_results_table,
    generate_spf_failures_table,
    generate_test_failures_by_variant_table,
    generate_test_failures_by_projects_table,
    generate_processing_failures_table,
    generate_exclusions_summary_csv,
    generate_filtering_results_csv,
    generate_spf_failures_csv,
    generate_test_failures_by_variant_csv,
    generate_test_failures_by_projects_csv,
    generate_processing_failures_by_cause_csv,
)
from teralizer.exports import save_latex_table, save_csv_data
from teralizer.plotting import setup_paper_style
from IPython.display import display

import pandas as pd

# Database connections
conn_dev = db_config.get_dev_engine()  # Main evaluation dataset
conn_test = db_config.get_test_engine()  # Extended dataset

# Configure paper style
setup_paper_style()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Overall Exclusions Summary

Analysis of included and excluded counts by variant and level (Test/Assertion/Generalization).

In [ ]:
# Get overall exclusions data
exclusions_raw = get_exclusions_summary_data(conn_dev)
exclusions_summary = compute_exclusion_percentages(exclusions_raw)

print("Overall exclusions by variant and level:")
display(
    exclusions_summary[
        [
            "variant",
            "Type",
            "Total",
            "included_count",
            "excluded_count",
            "included_pct",
            "excluded_pct",
        ]
    ]
)

# Generate LaTeX table
exclusions_table = generate_exclusions_summary_table(exclusions_summary)
print("\n=== LaTeX Table ===")
print(exclusions_table)

# Save LaTeX table
save_latex_table(exclusions_table, "tab-exclusions-summary")

# Generate and save CSV data
exclusions_csv = generate_exclusions_summary_csv(exclusions_summary)
csv_path = save_csv_data(
    exclusions_csv,
    "exclusions-summary-data",
    "Included and excluded counts by variant and level (Test/Assertion/Generalization)",
)

print(f"\nExclusions summary data exported to: {csv_path}")
print(f"Shape: {exclusions_csv.shape}")
print("Sample data:")
display(exclusions_csv.head())

## Filtering-Based Exclusions

Analysis of filtering results for tests, assertions, and generalizations by filter type and variant.

In [ ]:
# Get filtering exclusions data from main dataset
filtering_raw_main = get_filtering_exclusions_data(conn_dev)
filtering_main = compute_filtering_exclusions_summary(filtering_raw_main.copy())

print("Main dataset filtering results:")
display(
    filtering_main[
        [
            "variant",
            "Type",
            "filter_name",
            "total",
            "accept",
            "reject",
            "accept_pct",
            "reject_pct",
        ]
    ]
)

# Generate LaTeX table for main dataset
main_filtering_table = generate_filtering_results_table(
    filtering_main,
    "tab:exclusions-filtering",
    "Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.",
)

print("\n=== Main Dataset Filtering Results ===\n")
print(main_filtering_table)

# Save main dataset table
save_latex_table(main_filtering_table, "tab-exclusions-filtering")

# Export CSV data for main dataset
main_filtering_csv = generate_filtering_results_csv(filtering_main, "main")
csv_path = save_csv_data(
    main_filtering_csv,
    "exclusions-filtering-data",
    "Filtering results for tests, assertions, and generalizations by filter and variant",
)
print(f"Main filtering data exported to: {csv_path}")

In [ ]:
# Try to get extended dataset filtering results
try:
    filtering_raw_extended = get_filtering_exclusions_data(conn_test)
    filtering_extended = compute_filtering_exclusions_summary(
        filtering_raw_extended.copy()
    )

    print("Extended dataset filtering results:")
    display(
        filtering_extended[
            [
                "variant",
                "Type",
                "filter_name",
                "total",
                "accept",
                "reject",
                "accept_pct",
                "reject_pct",
            ]
        ]
    )

    # Generate LaTeX table for extended dataset
    extended_filtering_table = generate_filtering_results_table(
        filtering_extended,
        "tab:exclusions-filtering-extended",
        "Filtering results of the extended dataset for tests, assertions, and generalizations.",
    )

    print("\n=== Extended Dataset Filtering Results ===\n")
    print(extended_filtering_table)

    # Save extended dataset table
    save_latex_table(extended_filtering_table, "tab-exclusions-filtering-extended")

    # Export CSV data for extended dataset
    extended_filtering_csv = generate_filtering_results_csv(
        filtering_extended, "extended"
    )
    csv_path = save_csv_data(
        extended_filtering_csv,
        "exclusions-filtering-extended-data",
        "Extended dataset filtering results",
    )
    print(f"Extended filtering data exported to: {csv_path}")

except Exception as e:
    print(f"Extended dataset filtering not available: {e}")

## SPF Execution Failures

Analysis of Symbolic PathFinder execution failures by error type.

In [ ]:
# Get SPF failures data
spf_raw = get_spf_failures_data(conn_dev)
spf_categorized = compute_spf_error_categorization(spf_raw)

print("SPF execution failures by error type:")
display(spf_categorized)

# Generate LaTeX table
spf_table = generate_spf_failures_table(spf_categorized)
print("\n=== SPF Failures Table ===\n")
print(spf_table)

# Save LaTeX table
save_latex_table(spf_table, "tab-exclusions-spf")

# Generate and save CSV data
spf_csv = generate_spf_failures_csv(spf_categorized)
csv_path = save_csv_data(
    spf_csv, "exclusions-spf-data", "SPF execution failures by error type"
)

print(f"\nSPF errors data exported to: {csv_path}")
print(f"Shape: {spf_csv.shape}")
print("Sample data:")
display(spf_csv)

## Test Execution Failures

Analysis of test execution failures by exception type and generalization variant.

In [ ]:
# Get test execution data by variant
test_executions_by_variant_raw = get_test_executions_by_variant_data(conn_dev)
test_failures_by_variant = compute_test_failures_by_variant_summary(
    test_executions_by_variant_raw
)

print("Test execution failures by variant:")
display(
    test_failures_by_variant[
        ["variant", "total_executions", "null_pct", "too_many_filter_pct", "other_pct"]
    ]
)

# Generate LaTeX table
test_failures_by_variant_table = generate_test_failures_by_variant_table(
    test_failures_by_variant
)
print("\n=== Test Failures by Variant Table ===\n")
print(test_failures_by_variant_table)

# Save LaTeX table
save_latex_table(test_failures_by_variant_table, "tab-exclusions-test-fails-by-variant")

# Generate and save CSV data
test_failures_by_variant_csv = generate_test_failures_by_variant_csv(
    test_failures_by_variant
)
csv_path = save_csv_data(
    test_failures_by_variant_csv,
    "exclusions-test-fails-by-variant-data",
    "Test execution failures by variant",
)

print(f"\nTest failures by variant data exported to: {csv_path}")
print(f"Shape: {test_failures_by_variant_csv.shape}")
print("Sample data:")
display(test_failures_by_variant_csv.head())

## Test Execution Failures by Projects

Analysis of test execution failures grouped by projects using test generalization, showing failure type distribution across projects.

In [ ]:
# Get test failures data by projects
test_failures_by_projects_raw = get_test_failures_by_projects_data(conn_dev)
test_failures_by_projects = compute_test_failures_by_projects_summary(
    test_failures_by_projects_raw
)

print("Test execution failures by project (projects using test generalization):")
display(
    test_failures_by_projects[
        [
            "project_name",
            "total_executions",
            "null_pct",
            "too_many_filter_pct",
            "other_pct",
        ]
    ]
)

# Generate LaTeX table
test_failures_by_projects_table = generate_test_failures_by_projects_table(
    test_failures_by_projects
)
print("\n=== Test Failures by Projects Table ===\n")
print(test_failures_by_projects_table)

# Save LaTeX table
save_latex_table(
    test_failures_by_projects_table, "tab-exclusions-test-fails-by-project"
)

# Generate and save CSV data
test_failures_by_projects_csv = generate_test_failures_by_projects_csv(
    test_failures_by_projects
)
csv_path = save_csv_data(
    test_failures_by_projects_csv,
    "exclusions-test-fails-by-project-data",
    "Test execution failures by project (projects using test generalization)",
)

print(f"\nTest failures by projects data exported to: {csv_path}")
print(f"Shape: {test_failures_by_projects_csv.shape}")
print("Sample data:")
display(test_failures_by_projects_csv.head())

## Processing Pipeline Failures

Analysis of processing pipeline failures in the extended dataset, showing where projects fail in the processing stages.

In [ ]:
# Get processing failures data from extended dataset
try:
    processing_failures_raw = get_processing_failures_by_cause_data(conn_test)
    processing_failures = compute_processing_failures_by_stage_and_cause(
        processing_failures_raw
    )

    print("Processing failures by stage and cause:")
    display(processing_failures)

    # Generate LaTeX table
    processing_table = generate_processing_failures_table(processing_failures)
    print("\n=== Processing Failures Table ===\n")
    print(processing_table)

    # Save LaTeX table
    save_latex_table(processing_table, "tab-processing-failures")

    # Generate and save CSV data
    processing_csv = generate_processing_failures_by_cause_csv(processing_failures)
    csv_path = save_csv_data(
        processing_csv,
        "processing-failures-data",
        "Processing failures by stage and cause",
    )
    print(f"\nProcessing failures data exported to: {csv_path}")

except Exception as e:
    print(f"Processing pipeline analysis not available: {e}")
    print("This requires access to the extended dataset (conn_test).")

## Summary

This notebook has analyzed the limitations of the test generalization approach across multiple dimensions:

1. **Overall Exclusions**: Shows how many tests, assertions, and generalizations are filtered out by variant
2. **Filtering Results**: Details which specific filters cause exclusions and their effectiveness
3. **SPF Failures**: Categorizes symbolic execution failures by error type
4. **Test Failures**: Analyzes runtime test execution failures by variant
5. **Processing Pipeline**: Tracks where projects fail in the complete processing pipeline

All results have been exported as both LaTeX tables and CSV data for further analysis.

## RQ4 Analysis Complete

Generated outputs:
- `tab-exclusions-summary.tex` - Overall exclusions by variant and level
- `exclusions-summary-data.csv` - Overall exclusions data
- `tab-exclusions-filtering.tex` - Filtering results by filter type and variant
- `exclusions-filtering-data.csv` - Filtering results data
- `tab-exclusions-filtering-extended.tex` - Extended dataset filtering results
- `exclusions-filtering-extended-data.csv` - Extended dataset filtering data
- `tab-exclusions-spf.tex` - SPF execution failures by error type
- `exclusions-spf-data.csv` - SPF execution failures data
- `tab-exclusions-test-fails-by-variant.tex` - Test execution failures by variant
- `exclusions-test-fails-by-variant-data.csv` - Test execution failures by variant data
- `tab-exclusions-test-fails-by-project.tex` - Test execution failures by project
- `exclusions-test-fails-by-project-data.csv` - Test execution failures by project data
- `tab-processing-failures.tex` - Processing failures by stage and cause
- `processing-failures-data.csv` - Processing failures data